In [3]:
# =============================================================================
# CELL: Inspect play_by_play schema and sample rows
# Purpose: Confirm real column types and value shapes for play_by_play
#          BEFORE deciding on an explicit Spark schema. Read-only — no writes.
#          Row-count for this table already captured in the prior inventory
#          cell; not re-queried here to avoid a redundant full-table scan.
# =============================================================================

import sqlite3 #Importing this library becasuse the file is .sqlite
import pandas 

#File Path:
sqlite_path = "/Volumes/nba_data_kaggle/bronze/bronze_files/nba_data_kaggle_20260819.sqlite"

#Target Table to verify:
pbp_table_name = "play_by_play"

#Create connection with the .sqlite file:
# pbp = playbyplay 
conn_pbp = sqlite3.connect(sqlite_path)

#Execute SQL 
cursor_pbp = conn_pbp.cursor()


# --- Full column metadata: name, declared type, notnull flag, pk flag ---
cursor_pbp.execute(f"PRAGMA table_info({pbp_table_name});")
pbp_columns_info = cursor_pbp.fetchall()

print(f"play_by_play has {len(pbp_columns_info)} columns:\n")
for col in pbp_columns_info:
    # col = (cid, name, declared_type, notnull, default_value, pk)
    cid, col_name, declared_type, notnull, default_val, pk_flag = col
    print(f"  {col_name:<20} declared_type={declared_type:<12} notnull={bool(notnull)}  pk={bool(pk_flag)}")



OperationalError: unable to open database file

In [0]:
# =============================================================================
# CELL: play_by_play duplicate check and null profiling (SQL-side, no full load)
# Purpose: Confirm whether (game_id, eventnum) is a valid natural key, and
#          flag columns that are fully or near-fully null (VoidType risk).
#          Both aggregations run inside SQLite — only summary rows come back
#          to Python, never the 13.5M raw rows.
# =============================================================================

import sqlite3

sqlite_path = "/Volumes/nba_data_kaggle/bronze/bronze_files/nba_data_kaggle_20260819.sqlite"
pbp_table_name = "play_by_play"

conn_pbp_diag = sqlite3.connect(sqlite_path)
cursor_pbp_diag = conn_pbp_diag.cursor()

# --- 1. Candidate key check: total rows vs distinct (game_id, eventnum) ---
cursor_pbp_diag.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT game_id || '-' || eventnum) AS distinct_keys
    FROM {pbp_table_name};
""")
total_rows, distinct_keys = cursor_pbp_diag.fetchone()
print(f"Total rows:     {total_rows:,}")
print(f"Distinct keys:  {distinct_keys:,}")
print(f"Difference:     {total_rows - distinct_keys:,}")

# --- 2. Sample of actual duplicate keys, if any exist ---
cursor_pbp_diag.execute(f"""
    SELECT game_id, eventnum, COUNT(*) AS cnt
    FROM {pbp_table_name}
    GROUP BY game_id, eventnum
    HAVING COUNT(*) > 1
    LIMIT 20;
""")
dup_samples = cursor_pbp_diag.fetchall()
print(f"\nSample duplicate keys (up to 20): {len(dup_samples)} found")
for row in dup_samples:
    print(f"  game_id={row[0]}  eventnum={row[1]}  count={row[2]}")

# --- 3. Null count per column, built dynamically from PRAGMA column list ---
cursor_pbp_diag.execute(f"PRAGMA table_info({pbp_table_name});")
pbp_col_names = [col[1] for col in cursor_pbp_diag.fetchall()]

null_check_expr = ", ".join(
    f'SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS "{col}"'
    for col in pbp_col_names
)
cursor_pbp_diag.execute(f"SELECT {null_check_expr} FROM {pbp_table_name};")
null_counts = cursor_pbp_diag.fetchone()

conn_pbp_diag.close()

print(f"\nNull counts by column (out of {total_rows:,} rows):")
for col, null_count in zip(pbp_col_names, null_counts):
    pct = (null_count / total_rows) * 100
    flag = "  <-- fully null, VoidType risk" if null_count == total_rows else ""
    print(f"  {col:<28} {null_count:>10,}  ({pct:5.1f}%){flag}")

In [0]:
# =============================================================================
# CELL: Inspect actual duplicate rows in play_by_play
# Purpose: Determine whether (game_id, eventnum) duplicates are exact
#          row duplicates or distinct events sharing a coincidental key.
#          Pulls only the duplicate rows themselves — not a full table scan.
# =============================================================================

import sqlite3
import pandas as pd

sqlite_path = "/Volumes/nba_data_kaggle/bronze/bronze_files/nba_data_kaggle_20260819.sqlite"
pbp_table_name = "play_by_play"

conn_pbp_dupcheck = sqlite3.connect(sqlite_path)

# Pull a handful of duplicate keys first (cheap: SQL-side aggregation)
pdf_dup_keys = pd.read_sql_query(f"""
    SELECT game_id, eventnum, COUNT(*) AS cnt
    FROM {pbp_table_name}
    GROUP BY game_id, eventnum
    HAVING COUNT(*) > 1
    ORDER BY game_id, eventnum
    LIMIT 5;
""", conn_pbp_dupcheck)

print("Sample duplicate keys selected for inspection:")
print(pdf_dup_keys)

# For each of those 5 keys, pull the full rows so we can compare column by column
pdf_dup_rows_list = []
for _, key_row in pdf_dup_keys.iterrows():
    pdf_rows = pd.read_sql_query(f"""
        SELECT * FROM {pbp_table_name}
        WHERE game_id = '{key_row['game_id']}' AND eventnum = {key_row['eventnum']};
    """, conn_pbp_dupcheck)
    pdf_dup_rows_list.append(pdf_rows)

conn_pbp_dupcheck.close()

pdf_all_dup_rows = pd.concat(pdf_dup_rows_list, ignore_index=True)

# Show side by side, focused on the columns most likely to differ if these
# are genuinely different events rather than exact duplicates
cols_to_inspect = [
    "game_id", "eventnum", "eventmsgtype", "eventmsgactiontype",
    "period", "wctimestring", "pctimestring",
    "homedescription", "visitordescription", "neutraldescription",
    "player1_name", "player2_name"
]
print("\nFull rows for each duplicate key:")
print(pdf_all_dup_rows[cols_to_inspect].to_string())

# Quick check: are the two rows per key IDENTICAL across all columns, or do
# they differ? This tells us exact-duplicate vs distinct-event-same-key.
print("\nAre rows within each duplicate key fully identical (all columns)?")
for _, key_row in pdf_dup_keys.iterrows():
    subset = pdf_all_dup_rows[
        (pdf_all_dup_rows["game_id"] == key_row["game_id"]) &
        (pdf_all_dup_rows["eventnum"] == key_row["eventnum"])
    ]
    is_identical = subset.drop_duplicates().shape[0] == 1
    print(f"  game_id={key_row['game_id']} eventnum={key_row['eventnum']}: identical={is_identical}")